# Fotos -> Mesh 3D (.obj) con pycolmap



In [1]:
# Check GPU activa
!nvidia-smi

Fri Jul 31 04:44:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Instalar dependencias
# pycolmap-cuda12 trae binarios pre-compilados con CUDA 12, compatible con las GPUs T4 de Colab.
!pip install -q pycolmap-cuda12 open3d numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.2 MB/s eta 0:00:00


In [12]:
from google.colab import drive
from PIL import Image
import os
import shutil

name = "taj"
TARGET_SIZE = 1200

drive.mount("/content/drive")

src_folder = f"/content/drive/MyDrive/photos/{name}"
local_folder = "/content/photos_local"
small_folder = "/content/photos_small"

if not os.path.isdir(src_folder):
    raise FileNotFoundError(src_folder)

shutil.rmtree(local_folder, ignore_errors=True)
shutil.rmtree(small_folder, ignore_errors=True)

shutil.copytree(src_folder, local_folder)
os.makedirs(small_folder, exist_ok=True)

images = [
    f for f in os.listdir(local_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"{len(images)} fotos copiadas")

# images = images[:15]

for fname in images:
    src = os.path.join(local_folder, fname)
    dst = os.path.join(small_folder, fname)

    with Image.open(src) as img:
        img.thumbnail((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
        img.save(dst, quality=90)

print(f"{len(os.listdir(small_folder))} fotos redimensionadas")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
55 fotos copiadas
55 fotos redimensionadas


<!-- **Opción B (para muchas fotos / archivos grandes):** sube las fotos a una carpeta en tu Google Drive y monta Drive en vez de usar la celda anterior:
```python
from google.colab import drive
drive.mount('/content/drive')
photos_dir = '/content/drive/MyDrive/ruta/a/tus/fotos'  # ajusta la ruta
```
Y más abajo usa `photos_dir` en vez de `'photos'`. -->

In [19]:
# 4. Pipeline completo: SfM + reconstrucción densa + mesh
import sys
import shutil
from pathlib import Path

import numpy as np
import pycolmap


def run_sfm(database_path, image_dir, sparse_path, num_threads=-1, max_image_size=-1):
    print(f"[1/5] Extrayendo features SIFT (GPU)...")
    # extraction_options = {
    #     "num_threads": num_threads,
    #     "max_image_size": max_image_size,
    #     "use_gpu": True,
    # }

    extraction_options = {
    "num_threads": -1,
    "use_gpu": True,
    "sift": {"max_num_features": 4096},
    }

    pycolmap.extract_features(database_path, image_dir, extraction_options=extraction_options)

    print("[2/5] Matching exhaustivo entre imágenes (GPU)...")
    matching_options = {"num_threads": num_threads, "use_gpu": True}
    pycolmap.match_exhaustive(database_path, matching_options=matching_options)

    print("[3/5] Reconstrucción incremental (SfM)...")
    sparse_path.mkdir(parents=True, exist_ok=True)
    maps = pycolmap.incremental_mapping(database_path, image_dir, sparse_path)

    if not maps:
        sys.exit("La reconstrucción SfM falló: no se generó ningún modelo. "
                  "Revisa que las fotos tengan suficiente solapamiento/textura.")

    best_map = max(maps.values(), key=lambda r: r.num_reg_images())
    model_path = sparse_path / "0"
    model_path.mkdir(parents=True, exist_ok=True)
    best_map.write(model_path)
    print(f"    -> {best_map.num_reg_images()} imágenes registradas, "
          f"{len(best_map.points3D)} puntos 3D.")
    return best_map, model_path


def run_dense(image_dir, sparse_model_path, mvs_path):
    print("[4/5] Reconstrucción densa (patch_match_stereo, GPU)...")
    mvs_path.mkdir(parents=True, exist_ok=True)

    pycolmap.undistort_images(mvs_path, sparse_model_path, image_dir)
    pycolmap.patch_match_stereo(mvs_path, options={
        "max_image_size": 1200,
        "window_radius": 4,
        "num_iterations": 2,
        "geom_consistency": False,
    })

    fused_ply = mvs_path / "fused.ply"
    pycolmap.stereo_fusion(fused_ply, mvs_path, input_type="photometric", output_type="PLY")

    print(f"    -> Nube de puntos densa guardada en {fused_ply}")
    return fused_ply





# --- Configuración ---
image_dir = Path(small_folder)          # o photos_dir si usaste Google Drive
output_path = Path('out')


database_path = output_path / "database.db"
sparse_path = output_path / "sparse"
mvs_path = output_path / "mvs"
obj_path = output_path / "mesh.obj"

output_path.mkdir(parents=True, exist_ok=True)
if database_path.exists():
    database_path.unlink()
if sparse_path.exists():
    shutil.rmtree(sparse_path)

reconstruction, sparse_model_path = run_sfm(database_path, image_dir, sparse_path)
fused_ply = run_dense(image_dir, sparse_model_path, mvs_path)


[1/5] Extrayendo features SIFT (GPU)...
[2/5] Matching exhaustivo entre imágenes (GPU)...
[3/5] Reconstrucción incremental (SfM)...
    -> 55 imágenes registradas, 3683 puntos 3D.
[4/5] Reconstrucción densa (patch_match_stereo, GPU)...


KeyboardInterrupt: 

In [25]:
def mesh_from_point_cloud(ply_path, obj_path, depth=9):
    import open3d as o3d

    print("[5/5] Generando mesh (Poisson) y exportando a .obj...")
    pcd = o3d.io.read_point_cloud(str(ply_path))

    pcd, ind = pcd.remove_statistical_outlier(
      nb_neighbors=16,
      std_ratio=0.001
    )


    if len(pcd.points) == 0:
        sys.exit(f"La nube de puntos {ply_path} está vacía.")

    bbox = pcd.get_axis_aligned_bounding_box()
    diag = np.linalg.norm(bbox.get_extent())

    radius = diag * 0.01

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=radius,
            max_nn=30
        )
    )

    pcd.orient_normals_consistent_tangent_plane(30)

    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=depth)

    densities = np.asarray(densities)
    threshold = np.quantile(densities, 0.02)
    mesh.remove_vertices_by_mask(densities < threshold)
    mesh.compute_vertex_normals()

    obj_path.parent.mkdir(parents=True, exist_ok=True)
    o3d.io.write_triangle_mesh(str(obj_path), mesh)
    print(f"    -> Mesh final guardado en {obj_path}")

poisson_depth = 11                   # sube a 10-11 para más detalle (más lento)
mesh_from_point_cloud(fused_ply, obj_path, depth=poisson_depth)

print(f"\n✅ Listo: {obj_path}")

[5/5] Generando mesh (Poisson) y exportando a .obj...
[Open3D WARNING] Write OBJ can not include triangle normals.
    -> Mesh final guardado en out/mesh.obj

✅ Listo: out/mesh.obj


In [26]:
# Output de mesh resultante
from google.colab import files
files.download('out/mesh.obj')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>